# 13 — 2026 World Cup Simulation v2

Full 2026 WC simulation using the eloratings.net formula (v2) with Dixon-Coles correction.
10,000 Monte Carlo runs → champion probabilities for all 48 teams.

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import poisson
from collections import defaultdict
import pickle
import warnings
warnings.filterwarnings('ignore')

## 1. Load data & models

In [ ]:
fixtures = pd.read_csv(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\fixtures_clean.csv")
groups   = pd.read_csv(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\group_stages_clean.csv")
elo      = pd.read_csv(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\elo_all_teams_v2.csv")
ranks    = pd.read_csv(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\fifa_rankings.csv")

with open(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\home_model_v2.pkl", 'rb') as f:
    home_model = pickle.load(f)
with open(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\away_model_v2.pkl", 'rb') as f:
    away_model = pickle.load(f)

print("All files loaded!")
print(f"Fixtures: {len(fixtures)} matches")
print(f"Groups:   {len(groups)} teams")

In [ ]:
team_info = elo.merge(ranks, left_on='team', right_on='Nation', how='left').drop(columns='Nation')
team_info = team_info.rename(columns={'FIFA_2026_rank': 'fifa_rank'})
team_info = team_info[['team', 'elo', 'fifa_rank']]
team_dict = team_info.set_index('team').to_dict(orient='index')

print("Team lookup ready!")
print(team_dict['Spain'])
print(team_dict['Argentina'])

## 2. Match prediction with Dixon-Coles correction

In [ ]:
RHO = -0.13

def dc_correction(i, j, home_xg, away_xg, rho=RHO):
    if i == 0 and j == 0:   return 1 - home_xg * away_xg * rho
    elif i == 1 and j == 0: return 1 + away_xg * rho
    elif i == 0 and j == 1: return 1 + home_xg * rho
    elif i == 1 and j == 1: return 1 - rho
    else:                   return 1.0

def predict_match(home_team, away_team, neutral=True, tournament_weight=5.0):
    home_elo  = team_dict[home_team]['elo']
    away_elo  = team_dict[away_team]['elo']
    home_rank = team_dict[home_team]['fifa_rank']
    away_rank = team_dict[away_team]['fifa_rank']

    match = pd.DataFrame([{
        'home_elo':          home_elo,
        'away_elo':          away_elo,
        'elo_diff':          home_elo - away_elo,
        'neutral':           int(neutral),
        'tournament_weight': tournament_weight,
        'fifa_rank_diff':    home_rank - away_rank
    }])

    home_xg = home_model.predict(match)[0]
    away_xg = away_model.predict(match)[0]
    return home_xg, away_xg

def match_probabilities(home_xg, away_xg, max_goals=10):
    home_win, draw, away_win = 0, 0, 0
    for i in range(max_goals):
        for j in range(max_goals):
            p = poisson.pmf(i, home_xg) * poisson.pmf(j, away_xg) * dc_correction(i, j, home_xg, away_xg)
            if i > j:   home_win += p
            elif i == j: draw    += p
            else:        away_win += p
    return home_win, draw, away_win

# Test
home_xg, away_xg = predict_match('Spain', 'Argentina')
hw, d, aw = match_probabilities(home_xg, away_xg)
print(f"Spain vs Argentina (neutral)")
print(f"Spain win: {hw*100:.1f}%  Draw: {d*100:.1f}%  Argentina win: {aw*100:.1f}%")

## 3. Simulate a match / knockout match

In [ ]:
def simulate_match(home_team, away_team, neutral=True, tournament_weight=5.0):
    home_xg, away_xg = predict_match(home_team, away_team, neutral, tournament_weight)
    home_goals = np.random.poisson(home_xg)
    away_goals = np.random.poisson(away_xg)
    return home_goals, away_goals

def simulate_knockout_match(team1, team2, neutral=True):
    home_xg, away_xg = predict_match(team1, team2, neutral=neutral)
    hg = np.random.poisson(home_xg)
    ag = np.random.poisson(away_xg)

    # Extra time if draw
    if hg == ag:
        hg += np.random.poisson(home_xg * 0.33)
        ag += np.random.poisson(away_xg * 0.33)

    # Penalties if still level
    if hg == ag:
        he = team_dict[team1]['elo']
        ae = team_dict[team2]['elo']
        pen_prob = np.clip(0.5 + (he - ae) * 0.0001, 0.4, 0.6)
        if np.random.random() < pen_prob: hg += 1
        else:                             ag += 1

    winner = team1 if hg > ag else team2
    return winner, hg, ag

# Test
for _ in range(5):
    hg, ag = simulate_match('Spain', 'Argentina')
    print(f"Spain {hg} - {ag} Argentina")

## 4. Group stage simulation

In [ ]:
def simulate_group(group_name, group_teams, fixtures_df):
    table = {team: {'W': 0, 'D': 0, 'L': 0, 'GF': 0, 'GA': 0, 'Pts': 0} for team in group_teams}

    group_fixtures = fixtures_df[
        fixtures_df['home_team'].isin(group_teams) &
        fixtures_df['away_team'].isin(group_teams)
    ]

    for _, row in group_fixtures.iterrows():
        home, away = row['home_team'], row['away_team']
        hg, ag = simulate_match(home, away, neutral=row['neutral'])

        table[home]['GF'] += hg; table[home]['GA'] += ag
        table[away]['GF'] += ag; table[away]['GA'] += hg

        if hg > ag:
            table[home]['W'] += 1; table[home]['Pts'] += 3; table[away]['L'] += 1
        elif hg < ag:
            table[away]['W'] += 1; table[away]['Pts'] += 3; table[home]['L'] += 1
        else:
            table[home]['D'] += 1; table[away]['D'] += 1
            table[home]['Pts'] += 1; table[away]['Pts'] += 1

    table_df = pd.DataFrame(table).T
    table_df['GD'] = table_df['GF'] - table_df['GA']
    table_df = table_df.sort_values(['Pts', 'GD', 'GF'], ascending=False)
    table_df.index.name = 'team'
    return table_df.reset_index()

def simulate_all_groups(groups_df, fixtures_df):
    return {
        g: simulate_group(g, groups_df[groups_df['group'] == g]['nation'].tolist(), fixtures_df)
        for g in sorted(groups_df['group'].unique())
    }

# Test
group_a_teams = groups[groups['group'] == 'A']['nation'].tolist()
print(f"Group A: {group_a_teams}")
result = simulate_group('A', group_a_teams, fixtures)
print(result[['team', 'W', 'D', 'L', 'GF', 'GA', 'GD', 'Pts']])

## 5. Qualifiers — top 2 per group + best 8 third-placed

In [ ]:
def get_qualifiers(all_tables):
    group_winners, group_runners_up, third_placed = [], [], []

    for group_name, table in all_tables.items():
        group_winners.append({'team': table.iloc[0]['team'], 'group': group_name,
                              'pts': table.iloc[0]['Pts'], 'gd': table.iloc[0]['GD'], 'gf': table.iloc[0]['GF']})
        group_runners_up.append({'team': table.iloc[1]['team'], 'group': group_name,
                                 'pts': table.iloc[1]['Pts'], 'gd': table.iloc[1]['GD'], 'gf': table.iloc[1]['GF']})
        third_placed.append({'team': table.iloc[2]['team'], 'group': group_name,
                             'pts': table.iloc[2]['Pts'], 'gd': table.iloc[2]['GD'], 'gf': table.iloc[2]['GF']})

    third_df = pd.DataFrame(third_placed).sort_values(['pts', 'gd', 'gf'], ascending=False).head(8)

    qualifiers = (
        [t['team'] for t in group_winners] +
        [t['team'] for t in group_runners_up] +
        third_df['team'].tolist()
    )
    return qualifiers, group_winners, group_runners_up, third_df

# Test
all_tables = simulate_all_groups(groups, fixtures)
qualifiers, winners, runners, third = get_qualifiers(all_tables)
print(f"Qualifiers ({len(qualifiers)} teams):")
print(qualifiers)

## 6. Knockout rounds

In [ ]:
def simulate_knockout_rounds(teams):
    rounds = {
        'Round of 32': [], 'Round of 16': [], 'Quarter-finals': [],
        'Semi-finals': [], 'Final': [], 'Winner': None
    }

    current_round = teams.copy()
    round_names   = ['Round of 32', 'Round of 16', 'Quarter-finals', 'Semi-finals', 'Final']

    for round_name in round_names:
        next_round = []
        np.random.shuffle(current_round)
        for i in range(0, len(current_round), 2):
            t1, t2 = current_round[i], current_round[i+1]
            winner, hg, ag = simulate_knockout_match(t1, t2, neutral=True)
            next_round.append(winner)
            rounds[round_name].append(f"{t1} {hg}-{ag} {t2} → {winner}")
        current_round = next_round

    rounds['Winner'] = current_round[0]
    return rounds

# Test single simulation
knockout_results = simulate_knockout_rounds(qualifiers)
for round_name, matches in knockout_results.items():
    if round_name == 'Winner':
        print(f"\n🏆 WINNER: {matches}")
    else:
        print(f"\n{round_name}")
        for match in matches:
            print(f"  {match}")

## 7. Monte Carlo — 10,000 simulations

In [ ]:
N_SIMULATIONS = 10000
stage_counts  = defaultdict(lambda: defaultdict(int))
stages        = ['Round of 32', 'Round of 16', 'Quarter-finals', 'Semi-finals', 'Final', 'Winner']

print(f"Running {N_SIMULATIONS:,} simulations...")

for sim in range(N_SIMULATIONS):
    if sim % 1000 == 0:
        print(f"  {sim}/{N_SIMULATIONS}...")

    all_tables = simulate_all_groups(groups, fixtures)
    qualifiers, _, _, _ = get_qualifiers(all_tables)

    for team in qualifiers:
        stage_counts[team]['Round of 32'] += 1

    knockout_results = simulate_knockout_rounds(qualifiers)

    for round_name in ['Round of 16', 'Quarter-finals', 'Semi-finals', 'Final']:
        for match in knockout_results[round_name]:
            winner = match.split('→ ')[1].strip()
            stage_counts[winner][round_name] += 1

    stage_counts[knockout_results['Winner']]['Winner'] += 1

print("Done!")

## 8. Results

In [ ]:
all_teams = groups['nation'].tolist()

results = []
for team in all_teams:
    results.append({
        'Team':      team,
        'R32 %':     round(stage_counts[team]['Round of 32']     / N_SIMULATIONS * 100, 1),
        'R16 %':     round(stage_counts[team]['Round of 16']     / N_SIMULATIONS * 100, 1),
        'QF %':      round(stage_counts[team]['Quarter-finals']  / N_SIMULATIONS * 100, 1),
        'SF %':      round(stage_counts[team]['Semi-finals']     / N_SIMULATIONS * 100, 1),
        'Final %':   round(stage_counts[team]['Final']           / N_SIMULATIONS * 100, 1),
        'Winner %':  round(stage_counts[team]['Winner']          / N_SIMULATIONS * 100, 1),
    })

results_df = pd.DataFrame(results).sort_values('Winner %', ascending=False).reset_index(drop=True)
results_df.index += 1
print(results_df.to_string())

## 9. Save

In [ ]:
results_df.to_csv(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\simulation_results_v2.csv", index=False)
print("Saved simulation_results_v2.csv!")